# 03 - Model experiments
Five feature sets (word 1/1-2/1-3 grams, char 2-5, word+char) x three algorithms, group-aware CV on TRAIN, compared on VALIDATION. The three V1 baselines (word 1-2 grams) are flagged for comparison.

In [ ]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
import matplotlib; matplotlib.rcParams["figure.figsize"] = (6, 3.5)
import matplotlib.pyplot as plt, pandas as pd, numpy as np

In [ ]:
from src.data.prepare import prepare
from src.models.train_model import train_candidates, _select
out = prepare(seed=42, write=False, use_uci=True)
cands, fitted, cv = train_candidates(out["splits"], seed=42)
best, rule = _select(cands)
print("evaluation:", out["manifest"]["split"]["evaluation_status"]); print("selected:", best); print(rule)

In [ ]:
rows = [{"candidate": k, "V1 baseline": c["is_v1_baseline"], **{m: c["validation"].get(m) for m in ("pr_auc","roc_auc","precision","recall","f2","fpr","fnr","accuracy")}, "cv_f2": c["cv_f2_mean"]} for k, c in cands.items()]
pd.DataFrame(rows).sort_values("pr_auc", ascending=False).round(3)

In [ ]:
from src.models.evaluation import confusion_matrix_figure
for k in [best] + [f"{a}|word_1_2" for a in ("naive_bayes", "logistic_regression", "linear_svm")]:
    confusion_matrix_figure(cands[k]["validation"]["confusion_matrix"], k + " (validation)"); plt.show()